# 第15章 结构化产品与 ABS — 编程实验完整解答

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/albertandking/fixed-income/blob/main/notebooks/solutions/ch15_solutions.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/albertandking/fixed-income/main?labpath=notebooks/solutions/ch15_solutions.ipynb)

本 notebook 给出本章全部编程实验的完整可运行解答；联网（akshare）部分以注释/降级方式给出，离线也能跑通。


In [ ]:
# 自举单元：Colab/Binder 自动安装 fi；本地跳过。
import importlib.util, sys, subprocess
if importlib.util.find_spec('fi') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git','clone','--depth','1','https://github.com/albertandking/fixed-income.git','/content/fi-book'],check=False)
        subprocess.run([sys.executable,'-m','pip','install','-e','/content/fi-book'],check=False)
    else:
        print('提示：仓库根目录执行 `uv sync --extra all` 后运行本 notebook。')


## 编程实验 7：例13.1 + 图15-1（分层损失阶梯）


In [ ]:
import numpy as np
from fi import securitization as sz, plotting
plotting.use_chinese_style()
tranches = [('优先档',80),('夹层档',15),('次级档',5)]; sizes = dict(tranches)
for t in sz.attachment_detachment(tranches): print(f"{t['name']}: 附着{t['attach']*100:.0f}%-脱离{t['detach']*100:.0f}%")
pl = np.linspace(0,30,121)
fig, ax = plotting.new_axes()
for n,_ in tranches:
    ax.plot(pl, [sz.allocate_losses(x,tranches)[n]/sizes[n]*100 for x in pl], label=n)
ax.set_xlabel('资产池损失率 (%)'); ax.set_ylabel('该档损失率 (%)'); ax.set_title('分层与次级垫底'); ax.legend(); fig.tight_layout()


## 编程实验 8：违约率压力测试


In [ ]:
def stress(dr, rec=0.4):
    pl = 100*dr*(1-rec); a = sz.allocate_losses(pl, tranches); return {n: a[n]/sizes[n]*100 for n,_ in tranches}
fig, ax = plotting.new_axes(); drs = np.linspace(0,0.5,51)
for n,_ in tranches: ax.plot(drs*100, [stress(d)[n] for d in drs], label=n)
ax.set_xlabel('资产池违约率 (%)'); ax.set_ylabel('该档损失率 (%)'); ax.set_title('违约率压力(回收率40%)'); ax.legend(); fig.tight_layout()
for d in (0.1,0.2,0.35):
    s = stress(d); print(f'违约率{d*100:.0f}%: ' + '  '.join(f'{n}={s[n]:.0f}%' for n,_ in tranches))


## 编程实验 9：提前还款对各档现金流的影响（简化）


In [ ]:
# 资产池本来 4 期等额还本(每期25); 加入每期 20% 提前还款 -> 本金提前回笼
base = [25,25,25,25]; cpr = 0.20
bal = 100.0; pre = []
for sched in base:
    extra = bal*cpr; pay = min(bal, sched+extra); pre.append(pay); bal -= pay
print('正常还本:', base)
print('含提前还款:', [round(x,2) for x in pre], '  -> 本金提前回笼、加权期限缩短(MBS收缩风险)')
